# Satellite Configuration

[Satellites](../api_reference/sats/index.rst) are the basic unit of agent in the 
environment. Four things must be specified in subclasses of `Satellite`:

* The `observation_spec`, which defines the satellite's [observation](../api_reference/obs/index.rst).
* The `action_spec`, which defines the satellite's [actions](../api_reference/act/index.rst).
* The `dyn_type`,  which selects the underlying [dynamics model](../api_reference/sim/dyn/index.rst) used in simulation.
* The `fsw_type`,  which selects the underlying [flight software model](../api_reference/sim/fsw/index.rst).

A very simple satellite is defined below:

In [1]:
from typing import ClassVar

import numpy as np
from Basilisk.architecture import bskLogging

from bsk_rl import SatelliteTasking, act, data, obs, sats, scene
from bsk_rl.sim import dyn, fsw

bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)


class SimpleSatellite(sats.Satellite):
    observation_spec: ClassVar[list[obs.Observation]] = [obs.Time()]  # Passed as list of instantiated classes
    action_spec: ClassVar[list[act.Action]] = [act.Drift()]
    dyn_type = dyn.BasicDynamicsModel  # Passed as a type
    fsw_type = fsw.BasicFSWModel

## Setting Satellite Parameters

Without instantiating the satellite, parameters that can be set in the various models
can be inspected.

In [2]:
SimpleSatellite.default_sat_args()

{'hs_min': 0.0,
 'maxCounterValue': 4,
 'thrMinFireTime': 0.02,
 'desatAttitude': 'sun',
 'controlAxes_B': [1, 0, 0, 0, 1, 0, 0, 0, 1],
 'thrForceSign': 1,
 'K': 7.0,
 'Ki': -1,
 'P': 35.0,
 'utc_init': 'this value will be set by the world model',
 'batteryStorageCapacity': 288000.0,
 'storedCharge_Init': <function bsk_rl.sim.dyn.base.BasicDynamicsModel.<lambda>()>,
 'disturbance_vector': None,
 'dragCoeff': 2.2,
 'panelArea': 1.0,
 'basePowerDraw': 0.0,
 'wheelSpeeds': <function bsk_rl.sim.dyn.base.BasicDynamicsModel.<lambda>()>,
 'maxWheelSpeed': inf,
 'u_max': 0.2,
 'rwBasePower': 0.4,
 'rwMechToElecEfficiency': 0.0,
 'rwElecToMechEfficiency': 0.5,
 'panelEfficiency': 0.2,
 'nHat_B': array([ 0,  0, -1]),
 'mass': 330,
 'width': 1.38,
 'depth': 1.04,
 'height': 1.58,
 'sigma_init': <function bsk_rl.sim.dyn.base.DynamicsModel.<lambda>()>,
 'omega_init': <function bsk_rl.sim.dyn.base.DynamicsModel.<lambda>()>,
 'rN': None,
 'vN': None,
 'oe': <function bsk_rl.utils.orbital.random_orbit

These parameters can be overriden when instantiating the satellite through the `sat_args`
argument. 

In [3]:
sat = SimpleSatellite(
    name="SimpleSat-1",
    sat_args=dict(
        mass=300,  # Setting a constant value
        dragCoeff=lambda: np.random.uniform(2.0, 2.4),  # Setting a randomized value
    ),
)


Each time the simulation is reset, all of the function-based randomizers are called.

In [4]:
sat.generate_sat_args()  # Called by the environment on reset()
sat.sat_args

{'hs_min': 0.0,
 'maxCounterValue': 4,
 'thrMinFireTime': 0.02,
 'desatAttitude': 'sun',
 'controlAxes_B': [1, 0, 0, 0, 1, 0, 0, 0, 1],
 'thrForceSign': 1,
 'K': 7.0,
 'Ki': -1,
 'P': 35.0,
 'utc_init': 'this value will be set by the world model',
 'batteryStorageCapacity': 288000.0,
 'storedCharge_Init': 173341.28520129545,
 'disturbance_vector': None,
 'dragCoeff': 2.3874053937810316,
 'panelArea': 1.0,
 'basePowerDraw': 0.0,
 'wheelSpeeds': array([-1054.16036515,  -481.65055087,  -533.10200197]),
 'maxWheelSpeed': inf,
 'u_max': 0.2,
 'rwBasePower': 0.4,
 'rwMechToElecEfficiency': 0.0,
 'rwElecToMechEfficiency': 0.5,
 'panelEfficiency': 0.2,
 'nHat_B': array([ 0,  0, -1]),
 'mass': 300,
 'width': 1.38,
 'depth': 1.04,
 'height': 1.58,
 'sigma_init': array([0.99691085, 0.51778137, 0.86394875]),
 'omega_init': array([-6.92160346e-06, -9.95033589e-05,  6.36462406e-05]),
 'rN': None,
 'vN': None,
 'oe': <Basilisk.utilities.orbitalMotion.ClassicElements at 0x7f0a38cddfc0>,
 'mu': 3986004

As a result, each episode will have different randomized parameters:

In [5]:
for _ in range(3):
    sat.generate_sat_args()  # Called by the environment on reset()
    print("New value of dragCoeff:", sat.sat_args["dragCoeff"])

New value of dragCoeff: 2.10306833210873
New value of dragCoeff: 2.2089564512254807
New value of dragCoeff: 2.0643677261101208


## The Observation Specification

A variety of observation elements are available for satellites. Full documentation
can be [found here](../api_reference/obs/index.rst), but some commonly used elements
are explored below.

<div class="alert alert-info">

**Info:** In these examples, `obs_type=dict` is passed to the `Satellite` constructor
so that the observation is human readable. While some RL libraries support dictionary-based
observations, the default return type - the numpy array format - is more typically used.

</div>


### Satellite Properties

The most common type of observations is introspective; i.e. what is my current state?
Any `@property` in the `dyn_type` or `fsw_type` of the satellite can be accessed using
SatProperties.

In [6]:
class SatPropsSatellite(sats.Satellite):
    observation_spec: ClassVar[list[obs.Observation]] = [
        obs.SatProperties(
            # At a minimum, specify the property to observe
            dict(prop="wheel_speeds"),
            # You can specify the module to use for the observation, but it is not necessary
            # if only one module has for the property
            dict(prop="battery_charge_fraction", module="dynamics"), 
            # Properties can be normalized by some constant. This is generally desirable
            # for RL algorithms to keep values around [-1, 1].
            dict(prop="r_BN_P", norm=7e6),
        )
    ]
    action_spec: ClassVar[list[act.Action]] = [act.Drift()]
    dyn_type = dyn.BasicDynamicsModel
    fsw_type = fsw.BasicFSWModel

env = SatelliteTasking(
    satellite=SatPropsSatellite("PropSat-1", {}, obs_type=dict),
    log_level="CRITICAL",
)
observation, _ = env.reset()
observation

{'sat_props': {'wheel_speeds': array([146.08629281, 153.02093105,  51.04216061]),
  'battery_charge_fraction': 0.5961828112029987,
  'r_BN_P_normd': array([-0.38611178,  0.64945779,  0.62658179])}}

In some cases, you may want to access a bespoke property that is not natively implemented
in a model. To do that, simply extend the model with your desired property.

In [7]:
class BespokeFSWModel(fsw.BasicFSWModel):
    @property
    def meaning_of_life(self):
        return 42
    
class BespokeSatPropsSatellite(sats.Satellite):
    observation_spec: ClassVar[list[obs.Observation]] = [
        obs.SatProperties(dict(prop="meaning_of_life"))
    ]
    action_spec: ClassVar[list[act.Action]] = [act.Drift()]
    dyn_type = dyn.BasicDynamicsModel
    fsw_type = BespokeFSWModel

env = SatelliteTasking(
    satellite=BespokeSatPropsSatellite("BespokeSat-1", {}, obs_type=dict),
    log_level="CRITICAL",
)
observation, _ = env.reset()
observation

{'sat_props': {'meaning_of_life': 42.0}}

Alternatively, define the property with a function that takes the satellite object as an argument.

In [8]:
class CustomSatPropsSatellite(sats.Satellite):
    observation_spec: ClassVar[list[obs.Observation]] = [
        obs.SatProperties(dict(prop="meaning_of_life", fn=lambda sat: 42))
    ]
    action_spec: ClassVar[list[act.Action]] = [act.Drift()]
    dyn_type = dyn.BasicDynamicsModel
    fsw_type = fsw.BasicFSWModel

env = SatelliteTasking(
    satellite=CustomSatPropsSatellite("BespokeSat-1", {}, obs_type=dict),
    log_level="CRITICAL",
)
observation, _ = env.reset()
observation

{'sat_props': {'meaning_of_life': 42.0}}

### Opportunity Properties
Another common input to the observation is information about upcoming locations that 
are being accessed by the satellite. Currently, these include ground stations for
downlink and targets for imaging, but `OpportunityProperties` will work with any
location added by `add_location_for_access_checking`. In these examples, 

In [9]:
class OppPropsSatellite(sats.ImagingSatellite):
    observation_spec: ClassVar[list[obs.Observation]] = [
        obs.OpportunityProperties(
            # Properties can be added by some default names
            dict(prop="priority"), 
            # They can also be normalized
            dict(prop="opportunity_open", norm=5700.0),
            # Or they can be specified by an arbitrary function
            dict(fn=lambda sat, opp: opp["r_LP_P"] + 42),
            n_ahead_observe=3,
        )
    ]
    action_spec: ClassVar[list[act.Action]] = [act.Drift()]
    dyn_type = dyn.ImagingDynModel
    fsw_type = fsw.ImagingFSWModel

env = SatelliteTasking(
    satellite=OppPropsSatellite("OppSat-1", {}, obs_type=dict),
    scenario=scene.UniformTargets(1000),
    rewarder=data.UniqueImageReward(),
    log_level="CRITICAL",
)
observation, _ = env.reset()
observation

{'target': {'target_0': {'priority': 0.7710847185283478,
   'opportunity_open_normd': np.float64(0.0),
   'prop_2': array([ -162885.57089777, -6328471.97739335,   777211.17903239])},
  'target_1': {'priority': 0.5044838440261068,
   'opportunity_open_normd': np.float64(0.0),
   'prop_2': array([ -252240.95361725, -6329443.11574279,   744755.34697603])},
  'target_2': {'priority': 0.1290913200604028,
   'opportunity_open_normd': 0.014551007711235782,
   'prop_2': array([ -864728.84353565, -6313218.00784963,   274898.23463311])}}}


### Navigating the Observation

Usually, multiple observation types need to be composed to sufficiently represent the
environment for the learning agent. Simply add multiple observations to the observation
specification list to combine them in the observation.


In [10]:
class ComposedObsSatellite(sats.Satellite):
    observation_spec: ClassVar[list[obs.Observation]] = [
        obs.Eclipse(),
        obs.SatProperties(dict(prop="battery_charge_fraction"))
    ]
    action_spec: ClassVar[list[act.Action]] = [act.Drift()]
    dyn_type = dyn.BasicDynamicsModel
    fsw_type = fsw.BasicFSWModel

env = SatelliteTasking(
    satellite=ComposedObsSatellite("PropSat-1", {}, obs_type=dict),
    log_level="CRITICAL",
)
observation, _ = env.reset()
observation

{'eclipse': [np.float64(3330.0), np.float64(4290.0)],
 'sat_props': {'battery_charge_fraction': 0.8576184491787897}}


A few useful functions exist for inspecting the observation. The `observation_space`
property of the satellite and the environment return a Gym observation space to describe
the observation. In the single agent `SatelliteTasking` environment, these are the same.

<div class="alert alert-info">

**Info:** Here, we return to the `ndarray` default observation type.

</div>

In [11]:
env = SatelliteTasking(
    satellite=ComposedObsSatellite("PropSat-1", {}),
    log_level="CRITICAL",
)
(env.observation_space, env.unwrapped.satellite.observation_space)

(Box(-1e+16, 1e+16, (3,), float64), Box(-1e+16, 1e+16, (3,), float64))


With the flattened-vector type observation, it can be hard for the user to relate
elements to specific observations.


In [12]:
observation, _ = env.reset()
observation

array([5.67000000e+03, 2.01000000e+03, 8.70837668e-01])

The `observation_description` property can help the user understand what elements are 
present in the observation.

In [13]:
env.unwrapped.satellite.observation_description

[np.str_('eclipse[0]'),
 np.str_('eclipse[1]'),
 np.str_('sat_props.battery_charge_fraction')]


## The Action Specification

The [action specification](../api_reference/act/index.rst) works similarly to observation
specification. A list of actions is set in the class definition of the satellite.

In [14]:
class ActionSatellite(sats.Satellite):
    observation_spec: ClassVar[list[obs.Observation]] = [obs.Time()]
    action_spec: ClassVar[list[act.Action]] = [
        # If action duration is not set, the environment max_step_duration will be used;
        # however, being explicit is always preferable
        act.Charge(duration=120.0),
        act.Desat(duration=60.0),
        # One action can be included multiple time, if different settings are desired
        act.Charge(duration=600.0,),
    ]
    dyn_type = dyn.BasicDynamicsModel
    fsw_type = fsw.BasicFSWModel

env = SatelliteTasking(
    satellite=ActionSatellite("ActSat-1", {}, obs_type=dict),
    log_level="INFO",
)
env.reset()

# Try each action; index corresponds to the order of addition
_ =env.step(0)
_ =env.step(1)
_ =env.step(2)

2026-07-28 23:40:12,694                                WARNING    Creating logger for new env on PID=5195. Old environments in process may now log times incorrectly.


2026-07-28 23:40:12,696 gym                            INFO       Resetting environment with seed=514333259


2026-07-28 23:40:12,710 gym                            INFO       <0.00> Environment reset


2026-07-28 23:40:12,711 gym                            INFO       <0.00> === STARTING STEP ===


2026-07-28 23:40:12,711 sats.satellite.ActSat-1        INFO       <0.00> ActSat-1: action_charge tasked for 120.0 seconds


...

2026-07-28 23:40:12,752 sats.satellite.ActSat-1        INFO       <780.00> ActSat-1: timed termination at 780.0 for action_charge


2026-07-28 23:40:12,753 data.base                      INFO       <780.00> Total reward: {}


2026-07-28 23:40:12,753 comm.communication             INFO       <780.00> Optimizing data communication between all pairs of satellites


2026-07-28 23:40:12,753 sats.satellite.ActSat-1        INFO       <780.00> ActSat-1: Satellite ActSat-1 requires retasking


2026-07-28 23:40:12,755 gym                            INFO       <780.00> Step reward: 0.0


As with the observations, properties exist to help understand the actions available.

In [15]:
env.action_space

Discrete(3)

In [16]:
env.unwrapped.satellite.action_description

['action_charge', 'action_desat', 'action_charge']

Some actions take additional configurations, add multiple actions to the satellite, and/or
have "special" features that are useful for manually interacting with the environment. 
For example, the imaging action can add an arbitrary number of actions corresponding to
upcoming targets and process the name of a target directly instead of operating by
action index.

In [17]:
class ImageActSatellite(sats.ImagingSatellite):
    observation_spec: ClassVar[list[obs.Observation]] = [obs.Time()]
    action_spec: ClassVar[list[act.Action]] = [
        # Set the number of upcoming targets to consider
        act.Image(n_ahead_image=3)
    ]
    dyn_type = dyn.ImagingDynModel
    fsw_type = fsw.ImagingFSWModel

env = SatelliteTasking(
    satellite=ImageActSatellite("ActSat-2", {}),
    scenario=scene.UniformTargets(1000),
    rewarder=data.UniqueImageReward(),
    log_level="INFO",
)
env.reset()

env.unwrapped.satellite.action_description

2026-07-28 23:40:12,770                                WARNING    Creating logger for new env on PID=5195. Old environments in process may now log times incorrectly.


2026-07-28 23:40:12,771 gym                            INFO       Resetting environment with seed=589494983


2026-07-28 23:40:12,772 scene.targets                  INFO       Generating 1000 targets


2026-07-28 23:40:12,793 gym                            INFO       <0.00> Environment reset


['action_image_0', 'action_image_1', 'action_image_2']

Demonstrating the action overload feature, we task the satellite based on target name.
While this is not part of the official Gym API, we find it useful in certain cases.

In [18]:
target = env.unwrapped.satellite.find_next_opportunities(n=10)[9]["object"]
_ = env.step(target)

2026-07-28 23:40:12,798 sats.satellite.ActSat-2        INFO       <0.00> ActSat-2: Finding opportunity windows from 0.00 to 600.00 seconds


2026-07-28 23:40:12,819 sats.satellite.ActSat-2        INFO       <0.00> ActSat-2: Finding opportunity windows from 600.00 to 1200.00 seconds


2026-07-28 23:40:12,842 gym                            INFO       <0.00> === STARTING STEP ===


2026-07-28 23:40:12,843 act.discrete_actions           WARNING    <0.00> Action 'Target(tgt-894)' is not an integer. Will attempt to use compatible set_action_override method.


2026-07-28 23:40:12,844 sats.satellite.ActSat-2        INFO       <0.00> ActSat-2: Target(tgt-894) tasked for imaging


...

2026-07-28 23:40:12,922 sats.satellite.ActSat-2        INFO       <691.00> ActSat-2: imaged Target(tgt-894)


2026-07-28 23:40:12,923 data.base                      INFO       <691.00> Total reward: {'ActSat-2': 0.7000961226339221}


2026-07-28 23:40:12,923 comm.communication             INFO       <691.00> Optimizing data communication between all pairs of satellites


2026-07-28 23:40:12,923 sats.satellite.ActSat-2        INFO       <691.00> ActSat-2: Satellite ActSat-2 requires retasking


2026-07-28 23:40:12,924 gym                            INFO       <691.00> Step reward: 0.7000961226339221
